In [0]:
%sql
-- DIM_Fecha: Incluye fechas de asteroides y cometas
USE CATALOG nasa_dw;

CREATE TABLE IF NOT EXISTS nasa_dw.gold.dim_fecha
USING DELTA
COMMENT 'Dimensión temporal'
AS SELECT DISTINCT
    CAST(DATE_FORMAT(fecha_acercamiento, 'yyyyMMdd') AS INT) AS fecha_id,
    fecha_acercamiento                                        AS fecha,
    YEAR(fecha_acercamiento)                                  AS anio,
    QUARTER(fecha_acercamiento)                               AS trimestre,
    MONTH(fecha_acercamiento)                                 AS mes,
    DATE_FORMAT(fecha_acercamiento, 'MMMM')                   AS nombre_mes,
    WEEKOFYEAR(fecha_acercamiento)                            AS semana_anio,
    DAY(fecha_acercamiento)                                   AS dia_mes,
    DAYOFWEEK(fecha_acercamiento)                             AS dia_semana,
    DATE_FORMAT(fecha_acercamiento, 'EEEE')                   AS nombre_dia,
    CASE WHEN DAYOFWEEK(fecha_acercamiento) IN (1,7) THEN TRUE ELSE FALSE END AS es_fin_semana
FROM nasa_dw.silver.neo_clean WHERE 1=0;

MERGE INTO nasa_dw.gold.dim_fecha AS tgt
USING (
    -- Fechas de asteroides
    SELECT DISTINCT
        CAST(DATE_FORMAT(fecha_acercamiento, 'yyyyMMdd') AS INT) AS fecha_id,
        fecha_acercamiento                                        AS fecha,
        YEAR(fecha_acercamiento)                                  AS anio,
        QUARTER(fecha_acercamiento)                               AS trimestre,
        MONTH(fecha_acercamiento)                                 AS mes,
        DATE_FORMAT(fecha_acercamiento, 'MMMM')                   AS nombre_mes,
        WEEKOFYEAR(fecha_acercamiento)                            AS semana_anio,
        DAY(fecha_acercamiento)                                   AS dia_mes,
        DAYOFWEEK(fecha_acercamiento)                             AS dia_semana,
        DATE_FORMAT(fecha_acercamiento, 'EEEE')                   AS nombre_dia,
        CASE WHEN DAYOFWEEK(fecha_acercamiento) IN (1,7) THEN TRUE ELSE FALSE END AS es_fin_semana
    FROM nasa_dw.silver.neo_clean
    
    UNION
    
    -- Fechas de cometas
    SELECT DISTINCT
        CAST(DATE_FORMAT(fecha_acercamiento, 'yyyyMMdd') AS INT) AS fecha_id,
        fecha_acercamiento                                        AS fecha,
        YEAR(fecha_acercamiento)                                  AS anio,
        QUARTER(fecha_acercamiento)                               AS trimestre,
        MONTH(fecha_acercamiento)                                 AS mes,
        DATE_FORMAT(fecha_acercamiento, 'MMMM')                   AS nombre_mes,
        WEEKOFYEAR(fecha_acercamiento)                            AS semana_anio,
        DAY(fecha_acercamiento)                                   AS dia_mes,
        DAYOFWEEK(fecha_acercamiento)                             AS dia_semana,
        DATE_FORMAT(fecha_acercamiento, 'EEEE')                   AS nombre_dia,
        CASE WHEN DAYOFWEEK(fecha_acercamiento) IN (1,7) THEN TRUE ELSE FALSE END AS es_fin_semana
    FROM nasa_dw.silver.cometas_clean
) AS src
ON tgt.fecha_id = src.fecha_id
WHEN NOT MATCHED THEN INSERT *;